# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [39]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [40]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [41]:
# Write your code below.

import os
from glob import glob
from dotenv import load_dotenv


In [42]:
load_dotenv()
price_data_dir = os.getenv('PRICE_DATA')
print(f"PRICE_DATA in: {price_data_dir}")

if price_data_dir:
    print(f"Directory exists: {os.path.exists(price_data_dir)}")
else:
    print("ERROR: PRICE_DATA is None - not loaded from .env file")
    
parquet_files = glob(os.path.join(price_data_dir, '*.parquet'))
parquet_files = glob(os.path.join(price_data_dir, '**/*.parquet'), recursive=True)

print(f"Found {len(parquet_files)} parquet files:")
for file in parquet_files:
    print(file)

PRICE_DATA in: ../../05_src/data/prices/
Directory exists: True
Found 2838 parquet files:
../../05_src/data/prices/MDLX/MDLX_2020/part.0.parquet
../../05_src/data/prices/MDLX/MDLX_2020/part.1.parquet
../../05_src/data/prices/MDLX/MDLX_2018/part.0.parquet
../../05_src/data/prices/MDLX/MDLX_2018/part.1.parquet
../../05_src/data/prices/MDLX/MDLX_2016/part.0.parquet
../../05_src/data/prices/MDLX/MDLX_2016/part.1.parquet
../../05_src/data/prices/MDLX/MDLX_2017/part.0.parquet
../../05_src/data/prices/MDLX/MDLX_2017/part.1.parquet
../../05_src/data/prices/MDLX/MDLX_2019/part.0.parquet
../../05_src/data/prices/MDLX/MDLX_2019/part.1.parquet
../../05_src/data/prices/ENG/ENG_2012/part.0.parquet
../../05_src/data/prices/ENG/ENG_2012/part.1.parquet
../../05_src/data/prices/ENG/ENG_2015/part.0.parquet
../../05_src/data/prices/ENG/ENG_2015/part.1.parquet
../../05_src/data/prices/ENG/ENG_2014/part.0.parquet
../../05_src/data/prices/ENG/ENG_2014/part.1.parquet
../../05_src/data/prices/ENG/ENG_2013/part

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [58]:


import dask.dataframe as dd


df = dd.read_parquet(parquet_files)
print(f"Number of partitions: {df.npartitions}")
print(f"Columns: {df.columns.tolist()}")
print(f"Total rows: {len(df)}")

print(f"\nFirst few rows:")
print(df.head(5))

def add_lags(group):
    group = group.sort_values('date')
    group['Close_lag_1'] = group['Close'].shift(1)
    group['Adj_Close_lag_1'] = group['Adj_Close'].shift(1)
    return group

dd_feat = df.groupby('ticker').apply(add_lags, meta=df)

print(dd_feat.head(5))

def create_features(group):
    group = group.sort_values('Date')
    group['Close_lag_1'] = group['Close'].shift(1)
    group['Adj_Close_lag_1'] = group['Adj Close'].shift(1)
    group['returns'] = (group['Close'] / group['Close_lag_1']) - 1
    group['hi_lo_range'] = group['High'] - group['Low']
    return group


meta = df._meta.copy()
meta['Close_lag_1'] = 0.0
meta['Adj_Close_lag_1'] = 0.0
meta['returns'] = 0.0
meta['hi_lo_range'] = 0.0

dd_feat = df.groupby('ticker').apply(create_features, meta=meta)

df_feat = dd_feat.compute()

print("✓ Converted dd_feat to pandas DataFrame (df_feat)")
print(f"Type: {type(df_feat)}")
print(f"Shape: {df_feat.shape}")



Number of partitions: 2838
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year']
Total rows: 325161

First few rows:
            Date       Open       High        Low      Close  Adj Close  \
38739 1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
38740 1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
38741 1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
38742 1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
38743 1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   

           Volume source ticker  Year  
38739  62546300.0  A.csv      A  1999  
38740  15234100.0  A.csv      A  1999  
38741   6577800.0  A.csv      A  1999  
38742   5975600.0  A.csv      A  1999  
38743   4843200.0  A.csv      A  1999  
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, source, ticker, Year]
Index: []
✓ Converted dd_feat to pandas DataFram

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [60]:
# Write your code below.
import pandas as pd
df_feat = dd_feat.compute()

print("✓ Converted dd_feat to pandas DataFrame (df_feat)")
print(f"Type: {type(df_feat)}")
print(f"Shape: {df_feat.shape}")

df_feat = df_feat.reset_index(drop=True)

df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].rolling(10).mean().reset_index(level=0, drop=True)

print("\n✓ Added returns_ma_10 feature")
print(f"New columns: {df_feat.columns.tolist()}")
print("\nSample data:")
print(df_feat[['ticker', 'Date', 'returns', 'returns_ma_10']].head(20))

✓ Converted dd_feat to pandas DataFrame (df_feat)
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (325161, 14)

✓ Added returns_ma_10 feature
New columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'returns', 'hi_lo_range', 'returns_ma_10']

Sample data:
   ticker       Date   returns  returns_ma_10
0     JHG 2017-05-30       NaN            NaN
1     JHG 2017-05-31  0.022222            NaN
2     JHG 2017-06-01  0.085038            NaN
3     JHG 2017-06-02 -0.019151            NaN
4     JHG 2017-06-05 -0.006008            NaN
5     JHG 2017-06-06 -0.045029            NaN
6     JHG 2017-06-07  0.003165            NaN
7     JHG 2017-06-08  0.002524            NaN
8     JHG 2017-06-09 -0.002203            NaN
9     JHG 2017-06-12  0.018291            NaN
10    JHG 2017-06-13  0.013936       0.007279
11    JHG 2017-06-14  0.038180       0.008874
12    JHG 2017-06-15 -0.025890      -0.002219
13    JHG 2017-

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it is possible to do it in DASK without coverting to pandas
Yes, it would have been better to do it in Dask as it has several advantage such as it doesn't need to load the entire dataset at once, it is faster and more reliable for large data.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.